In [1]:
ai_insights_df = spark.table(
    "demo.silver.ai_extracted_insights"
)

reference_materials_df = spark.table(
    "demo.silver.reference_materials"
)

print("AI insights:", ai_insights_df.count())
print("Reference materials:", reference_materials_df.count())

ai_insights_df.select(
    "insight_id",
    "event_id",
    "user_id",
    "dynamic_concept_name",
    "validation_status"
).show(truncate=False)

reference_materials_df.select(
    "reference_id",
    "title",
    "topic",
    "domain",
    "reliability_level"
).show(truncate=False)

AI insights: 9
Reference materials: 6
+----------------------------------------------------------------+--------+--------+--------------------+-----------------+
|insight_id                                                      |event_id|user_id |dynamic_concept_name|validation_status|
+----------------------------------------------------------------+--------+--------+--------------------+-----------------+
|a812e708876c32b4bb8d415842bf2de7a9114b7b88efe690f0c36a66875f65df|evt_0001|user_001|Operating Systems   |pending          |
|66317e7e2b57674ab50b4418ee576ccc39e490ca3f3f5009cd3ad2c3d474a3fe|evt_0001|user_001|Virtual Memory      |pending          |
|a6cb2c3a4bd19f4a389472160bd13969533dbe49e042f7855ab5ac61c66bfab6|evt_0001|user_001|Page Fault          |pending          |
|45cfab597ad270f7598c001b68a3495d488b397fd4171a0aa829f53ced7cf769|evt_0003|user_002|Programming         |pending          |
|36c747a8f354544cb0e51c4a61510305b145fe1b0681535403c1329db589baa2|evt_0003|user_002|Recursion 

In [2]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    concat_ws,
    coalesce,
    lit
)

insights_prepared_df = (
    ai_insights_df
    .select(
        "insight_id",
        "event_id",
        "user_id",
        "session_id",
        "dynamic_concept_name",
        "extracted_at"
    )
    .withColumn(
        "concept_normalized",
        lower(trim(col("dynamic_concept_name")))
    )
)

references_prepared_df = (
    reference_materials_df
    .filter(col("is_active") == True)
    .select(
        "reference_id",
        "domain",
        "title",
        "topic",
        "content_text",
        "reliability_level"
    )
    .withColumn(
        "reference_text",
        lower(
            concat_ws(
                " ",
                coalesce(col("domain"), lit("")),
                coalesce(col("topic"), lit("")),
                coalesce(col("title"), lit("")),
                coalesce(col("content_text"), lit(""))
            )
        )
    )
)

validation_candidates_df = (
    insights_prepared_df
    .crossJoin(references_prepared_df)
)

print(
    "Candidate comparisons:",
    validation_candidates_df.count()
)

Candidate comparisons: 54


In [5]:
from pyspark.sql.functions import (
    col,
    when,
    lit,
    expr
)

scored_candidates_df = (
    validation_candidates_df

    .withColumn(
        "title_match",
        when(
            expr(
                "instr(lower(coalesce(title, '')), concept_normalized) > 0"
            ),
            lit(1.0)
        ).otherwise(lit(0.0))
    )

    .withColumn(
        "topic_match",
        when(
            expr(
                "instr(lower(coalesce(topic, '')), concept_normalized) > 0"
            ),
            lit(1.0)
        ).otherwise(lit(0.0))
    )

    .withColumn(
        "content_match",
        when(
            expr(
                """
                instr(
                    lower(coalesce(content_text, '')),
                    concept_normalized
                ) > 0
                """
            ),
            lit(1.0)
        ).otherwise(lit(0.0))
    )

    .withColumn(
        "semantic_match_score",
        (
            col("title_match") * 0.5
            + col("topic_match") * 0.3
            + col("content_match") * 0.2
        ).cast("float")
    )
)

scored_candidates_df.select(
    "insight_id",
    "dynamic_concept_name",
    "reference_id",
    "title",
    "topic",
    "title_match",
    "topic_match",
    "content_match",
    "semantic_match_score"
).orderBy(
    "dynamic_concept_name",
    col("semantic_match_score").desc()
).show(100, truncate=False)

+----------------------------------------------------------------+--------------------+-------------+---------------------------------+-----------------+-----------+-----------+-------------+--------------------+
|insight_id                                                      |dynamic_concept_name|reference_id |title                            |topic            |title_match|topic_match|content_match|semantic_match_score|
+----------------------------------------------------------------+--------------------+-------------+---------------------------------+-----------------+-----------+-----------+-------------+--------------------+
|1e03e8cbacf3d9cbfae14755113f800a3e50e69859b3dd8bcb902458d265dfa9|Base Case           |reference_004|Recursion and the Base Case      |Programming      |1.0        |0.0        |1.0          |0.7                 |
|1e03e8cbacf3d9cbfae14755113f800a3e50e69859b3dd8bcb902458d265dfa9|Base Case           |reference_001|Page Fault Definition            |Operating Sys

In [12]:
from pyspark.sql.functions import (
    col,
    when,
    lit,
    current_timestamp,
    sha2,
    concat_ws,
    round as spark_round
)

validated_learning_insights_df = (
    best_reference_match_df

    # שומרים כ-double כדי שהשוואת הספים תהיה יציבה
    .withColumn(
        "semantic_score_rounded",
        spark_round(
            col("semantic_match_score").cast("double"),
            2
        )
    )

    .withColumn(
        "source_reliability_score",
        when(col("reliability_level") == "official", lit(1.0))
        .when(col("reliability_level") == "approved", lit(0.8))
        .when(col("reliability_level") == "external", lit(0.6))
        .otherwise(lit(0.4))
    )

    .withColumn(
        "reliability_score_rounded",
        spark_round(
            (
                col("semantic_score_rounded") * 0.7
                + col("source_reliability_score") * 0.3
            ),
            2
        )
    )

    .withColumn(
        "contradiction_flag",
        lit(False)
    )

    .withColumn(
        "validation_status",
        when(
            col("semantic_score_rounded") >= lit(0.7),
            lit("validated")
        )
        .when(
            col("semantic_score_rounded") >= lit(0.3),
            lit("weak_match")
        )
        .otherwise(
            lit("no_reference")
        )
    )

    .withColumn(
        "validation_time",
        current_timestamp()
    )

    .withColumn(
        "validation_id",
        sha2(
            concat_ws(
                "||",
                col("insight_id"),
                col("reference_id")
            ),
            256
        )
    )

    .withColumn(
        "validation_notes",
        concat_ws(
            " | ",
            lit("Baseline rule-based validation"),
            concat_ws(
                "",
                lit("Matched reference: "),
                col("title")
            ),
            concat_ws(
                "",
                lit("Source reliability: "),
                col("reliability_level")
            )
        )
    )

    # רק לאחר קביעת הסטטוס ממירים ל-float
    .withColumn(
        "semantic_match_score",
        col("semantic_score_rounded").cast("float")
    )

    .withColumn(
        "reliability_score",
        col("reliability_score_rounded").cast("float")
    )

    .select(
        "validation_id",
        "insight_id",
        "reference_id",
        "user_id",
        "session_id",
        "dynamic_concept_name",
        "validation_time",
        "semantic_match_score",
        "reliability_score",
        "contradiction_flag",
        "validation_notes",
        "validation_status"
    )
)

In [13]:
validated_learning_insights_df.select(
    "dynamic_concept_name",
    "semantic_match_score",
    "reliability_score",
    "validation_status"
).orderBy(
    col("semantic_match_score").desc(),
    col("dynamic_concept_name")
).show(truncate=False)

+--------------------+--------------------+-----------------+-----------------+
|dynamic_concept_name|semantic_match_score|reliability_score|validation_status|
+--------------------+--------------------+-----------------+-----------------+
|Base Case           |0.7                 |0.73             |validated        |
|Page Fault          |0.7                 |0.79             |validated        |
|Process Memory      |0.7                 |0.79             |validated        |
|Recursion           |0.7                 |0.73             |validated        |
|Virtual Memory      |0.7                 |0.79             |validated        |
|Memory Layout       |0.5                 |0.65             |weak_match       |
|Operating Systems   |0.3                 |0.51             |weak_match       |
|Operating Systems   |0.3                 |0.51             |weak_match       |
|Programming         |0.3                 |0.45             |weak_match       |
+--------------------+------------------

In [14]:
validated_learning_insights_df.groupBy(
    "validation_status"
).count().show()

+-----------------+-----+
|validation_status|count|
+-----------------+-----+
|        validated|    5|
|       weak_match|    4|
+-----------------+-----+



In [15]:
print(
    "Total validated rows:",
    validated_learning_insights_df.count()
)

print(
    "Distinct validation IDs:",
    validated_learning_insights_df
    .select("validation_id")
    .distinct()
    .count()
)

Total validated rows: 9
Distinct validation IDs: 9


In [16]:
spark.sql("""
DELETE FROM demo.silver.validated_learning_insights
""")

DataFrame[]

In [17]:
validated_learning_insights_df.writeTo(
    "demo.silver.validated_learning_insights"
).append()

In [18]:
spark.sql("""
SELECT
    validation_status,
    COUNT(*) AS row_count
FROM demo.silver.validated_learning_insights
GROUP BY validation_status
ORDER BY validation_status
""").show()

+-----------------+---------+
|validation_status|row_count|
+-----------------+---------+
|        validated|        5|
|       weak_match|        4|
+-----------------+---------+



In [19]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT validation_id) AS distinct_validation_ids
FROM demo.silver.validated_learning_insights
""").show()

+----------+-----------------------+
|total_rows|distinct_validation_ids|
+----------+-----------------------+
|         9|                      9|
+----------+-----------------------+

